# Bài 3: Mô hình phân lớp MNIST bằng Neural Network (Keras) với k-fold cross validation

Notebook này thực hiện **câu d** và **câu e**. Để chạy được hai câu này, bên dưới có một ô chuẩn bị dữ liệu ngắn (đọc dữ liệu và chuẩn hóa pixel về đoạn [0,1], tương ứng câu a, b, c).

## Chuẩn bị dữ liệu (cần cho câu d, e)

In [1]:
print("Họ và tên: Lê Quốc Trung - MSSV: 6451071082")
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import KFold

# Đọc dữ liệu train, test
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print("Kích thước train:", train_images.shape, train_labels.shape)
print("Kích thước test:", test_images.shape, test_labels.shape)

# Chuẩn hóa giá trị pixel về đoạn [0,1]
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0

Họ và tên: Lê Quốc Trung - MSSV: 6451071082
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Kích thước train: (60000, 28, 28) (60000,)
Kích thước test: (10000, 28, 28) (10000,)


## d) Định nghĩa hàm `create_model`

- Model là Neural Network (Flatten → Dense 128 ReLU → Dense 10 Softmax).
- Biên dịch mô hình với optimizer: `adam`; loss: `sparse_categorical_crossentropy`; metric: `accuracy`.

In [2]:
print("Họ và tên: Lê Quốc Trung - MSSV: 6451071082")

def create_model():
    model = keras.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

create_model().summary()

Họ và tên: Lê Quốc Trung - MSSV: 6451071082


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

## e) Huấn luyện mô hình bằng k-fold cross validation (k = 10)

Trong quá trình huấn luyện, in ra loss và accuracy của từng fold và lưu lại mô hình sau mỗi fold.

*Hướng dẫn: Sử dụng `KFold` trong sklearn để định nghĩa kfold với `n_splits=10, shuffle=True, random_state=42`.*

Với mỗi fold: `train_index, val_index in kfold.split(train_images)`
- Gán model là kết quả của hàm `create_model()`
- Gán `X_train_fold, X_val_fold, y_train_fold, y_val_fold`
- Huấn luyện mô hình dựa trên tập train `(X_train_fold, y_train_fold)`, đánh giá với tập validation `(X_val_fold, y_val_fold)`; epoch: 5; batch_size: 32
- Lưu model huấn luyện được vào mảng `models`
- Tính và in accuracy và loss của fold. Lưu vào mảng `accuracy_per_fold[]` và `loss_per_fold[]`

In [3]:
print("Họ và tên: Lê Quốc Trung - MSSV: 6451071082")

# Định nghĩa k-fold với k = 10
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

models = []              # lưu các mô hình đã huấn luyện
accuracy_per_fold = []   # accuracy của từng fold
loss_per_fold = []       # loss của từng fold

fold_no = 1
for train_index, val_index in kfold.split(train_images):
    print(f"\n===== Fold {fold_no}/10 =====")

    # Gán model là kết quả của hàm create_model()
    model = create_model()

    # Gán dữ liệu train/validation của fold
    X_train_fold, X_val_fold = train_images[train_index], train_images[val_index]
    y_train_fold, y_val_fold = train_labels[train_index], train_labels[val_index]

    # Huấn luyện trên tập train, đánh giá với tập validation
    model.fit(X_train_fold, y_train_fold,
              epochs=5, batch_size=32,
              validation_data=(X_val_fold, y_val_fold))

    # Lưu model huấn luyện được vào mảng models
    models.append(model)

    # Tính và in accuracy, loss của fold
    loss, accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    print(f"Fold {fold_no}: loss = {loss:.4f}, accuracy = {accuracy:.4f}")
    accuracy_per_fold.append(accuracy)
    loss_per_fold.append(loss)

    fold_no += 1

print("\n===== Kết quả tổng hợp =====")
for i in range(len(accuracy_per_fold)):
    print(f"Fold {i+1}: loss = {loss_per_fold[i]:.4f}, accuracy = {accuracy_per_fold[i]:.4f}")
print(f"Trung bình: loss = {np.mean(loss_per_fold):.4f}, accuracy = {np.mean(accuracy_per_fold):.4f}")


Họ và tên: Lê Quốc Trung - MSSV: 6451071082

===== Fold 1/10 =====
Epoch 1/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9231 - loss: 0.2706 - val_accuracy: 0.9590 - val_loss: 0.1410
Epoch 2/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9649 - loss: 0.1199 - val_accuracy: 0.9695 - val_loss: 0.1022
Epoch 3/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9751 - loss: 0.0810 - val_accuracy: 0.9765 - val_loss: 0.0850
Epoch 4/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9816 - loss: 0.0608 - val_accuracy: 0.9757 - val_loss: 0.0802
Epoch 5/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9855 - loss: 0.0473 - val_accuracy: 0.9778 - val_loss: 0.0738
Fold 1: loss = 0.0738, accuracy = 0.9778

===== Fold 2/10 =====
Epoch 1/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9228 - loss: 0.2700 - val_accuracy: 0.9557 - val_loss: 0.1530
Epoch 2/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9642 - loss: 0.1205 - val_ac